# Homework - Module 05

The goal of this homework is to familiarize users with monitoring for ML batch services, using PostgreSQL database to store metrics and Grafana to visualize them. For that, we'll use [the NYC taxi dataset](https://www1.nyc.gov/site/tlc/about/tlc-trip-record-data.page), more specifically the **Green** Taxi Trip Records dataset.

### Question 1. Prepare the dataset

We will start with the notebook `baseline_model_nyc_taxi_data.ipynb`. Let's download the March 2024 Green Taxi data to simulate a production usage of a taxi trip duration prediction service.

In [1]:
# Necessary import
import requests # for downloading data from the internet, enabling reuse in pipelines
import datetime # for time-related operations
import pandas as pd # for data manipulation

from evidently import DataDefinition
from evidently import Dataset
from evidently import Report
from evidently.metrics import ValueDrift, DriftedColumnsCount, MissingValueCount

from joblib import load, dump # for model saving/loading
from tqdm import tqdm # for progress bars during downloads

# For linear regression modeling and evaluation
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

In [2]:
# create folder to store datasets and reference data
! mkdir data

In [3]:
# Define a list of tuples with a filename and its corresponding local save path in the data folder
files = [('green_tripdata_2024-03.parquet', './data')]

print("Download file:")
# Iterate through the list to download files from URLs 
# dynamically generated for January and February 2022 green taxi trip data
for file, path in files:
    url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/{file}"
    # Streaming download
    resp = requests.get(url, stream = True)
    save_path = f"{path}/{file}" # path for saving files
    with open(save_path, "wb") as handle:
        for data in tqdm(resp.iter_content(),
                        desc = f"{file}",
                        postfix = f"save to {save_path}",
                        total = int(resp.headers["Content-Length"])):
            handle.write(data) # writing the data

Download file:


green_tripdata_2024-03.parquet: 100%|██████████| 1372372/1372372 [00:04<00:00, 302150.54it/s, save to ./data/green_tripdata_2024-03.parquet]


In [4]:
# Read he January data set
march_data = pd.read_parquet('data/green_tripdata_2024-03.parquet')
# Data shape
print(f"Shape of the March Green Taxi Trip Data: {march_data.shape}.")

Shape of the March Green Taxi Trip Data: (57457, 20).


The data set has `57457` rows.

### Question 2. Metric

Let's expand the number of data quality metrics we’d like to monitor! We will add one metric of our choice and a quantile value for the `"fare_amount"` column (`quantile=0.5`).

In [6]:
# Necessary import
from evidently.metrics import QuantileValue

# Build a report
report = Report(metrics = [
    ValueDrift(column = 'prediction'),
    DriftedColumnsCount(),
    MissingValueCount(column = 'prediction'),
    QuantileValue(column = "fare_amount", quantile = 0.5)
]
)

The chosen metric is the `QuantileValue` metric.

### Question 3. Monitoring

Let’s start monitoring. We will run expanded monitoring for a new batch of data (March 2024). 

In [7]:
# Data labeling / Selecting features
target = "duration_min"
num_features = ["passenger_count", "trip_distance", "fare_amount", "total_amount"]
cat_features = ["PULocationID", "DOLocationID"]

In [8]:
# Create target variable as trip duration 
march_data[target] = march_data.lpep_dropoff_datetime - march_data.lpep_pickup_datetime
# Make sure the duration is in minutes
march_data.duration_min = march_data.duration_min.apply(lambda td : float(td.total_seconds())/60)

In [9]:
# Filter out outliers
march_data = march_data[(march_data.duration_min >= 0) & (march_data.duration_min <= 60)]
march_data = march_data[(march_data.passenger_count > 0) & (march_data.passenger_count <= 8)]

In [10]:
# Set the data definition for column mapping
data_definition = DataDefinition(numerical_columns = num_features + ['prediction'], categorical_columns = cat_features)

In [17]:
# Initialize median lists
median_list = []

for day in range(1, 32):
    day_start = datetime.datetime(2024, 3, day)
    day_end = day_start + datetime.timedelta(days=1)

    day_data = march_data[
        (march_data.lpep_pickup_datetime >= day_start) &
        (march_data.lpep_pickup_datetime < day_end)
    ]

    if len(day_data) == 0:
        continue

    current_dataset = Dataset.from_pandas(day_data, data_definition)

    report = Report(metrics = [
        QuantileValue(column = "fare_amount", quantile = 0.5)
    ])

    result = report.run(current_data = current_dataset).dict()
    median = result['metrics'][0]['value']
    median_list.append(median)

print(f"Maximal median: {max(median_list)}")

Maximal median: 14.2


The maximum value of metric `quantile = 0.5` on the `"fare_amount"` column during March 2024 (calculated daily) is `14.2`.


### Question 4. Dashboard

Finally, let’s add panels with new added metrics to the dashboard. After we customize the dashboard let's save a dashboard config, so that we can access it later.

> Hint: click on “Save dashboard” to access JSON configuration of the dashboard. This configuration should be saved locally.

The dashboard config file should be placed in: `project_folder/config`  (05-monitoring/config).

---